# Pandas Pivot Tables & Melt

## Required imports

In [7]:
import pandas as pd
import numpy as np

## Long vs Wide Table

Data tables that are
- wide, are more human friendly and are easier to read
- long tables are easier for computers to process

## `pd.pivot_tables()`

(2026-07-19T15:43:05+07:00 - AI Generated with Claude Sonnet 4.6 (Thinking))

`pandas.pivot_table` reshapes data by aggregating values across two axes (rows and columns), similar to Excel's Pivot Table feature.

### Syntax

```python
pd.pivot_table(data, values=None, index=None, columns=None, aggfunc='mean', fill_value=None, margins=False, dropna=True, margins_name='All', observed=False, sort=True)
```

### Key Parameters

|Parameter|	Description|
|--|--|
|data|	DataFrame to pivot|
|values|	Column(s) to aggregate|
|index|	Column(s) to group as rows|
|columns|	Column(s) to group as column headers|
|aggfunc|	Aggregation function: 'mean', 'sum', 'count', 'min', 'max', or a list/dict|
|fill_value|	Replace NaN with this value|
|margins|	Add row/column totals (All)|
|dropna|	Drop columns with all NaN entries|

### Basic Examples

In [2]:
product_region_sales_df = pd.DataFrame({
    'Region': ['North', 'North', 'South', 'South', 'East'],
    'Product': ['A', 'B', 'A', 'B', 'A'],
    'Sales': [100, 200, 150, 250, 120]
})

#### Default aggregation is mean

In [3]:
pd.pivot_table(product_region_sales_df, values='Sales', index='Region', columns='Product')

Product,A,B
Region,,
East,120.0,NaN
North,100.0,200.0
South,150.0,250.0


#### Using sum with fill_value

In [4]:
pd.pivot_table(product_region_sales_df, values='Sales', index='Region', columns='Product', aggfunc='sum', fill_value=0)

Product,A,B
Region,,
East,120,0
North,100,200
South,150,250


#### Adding totals with `margins=True`

In [5]:
pd.pivot_table(product_region_sales_df, values='Sales', index='Region', columns='Product', aggfunc='sum', fill_value=0, margins=True)

Product,A,B,All
Region,,,
East,120,0,120
North,100,200,300
South,150,250,400
All,370,450,820


#### Multiple aggfunc

In [6]:
pd.pivot_table(product_region_sales_df, values='Sales', index='Region', aggfunc=['sum', 'mean', 'count'])

,sum,mean,count
,Sales,Sales,Sales
Region,,,
East,120,120.0,1
North,300,150.0,2
South,400,200.0,2


### Real-life Example: E-commerce Sales Analysis

In [9]:
orders = pd.DataFrame({
    'Month':    ['Jan', 'Jan', 'Feb', 'Feb', 'Mar', 'Mar', 'Mar'],
    'Category': ['Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Clothing', 'Food'],
    'Region':   ['US', 'EU', 'US', 'US', 'EU', 'EU', 'US'],
    'Revenue':  [5000, 1200, 4800, 900, 6200, 1500, 800],
    'Units':    [50, 120, 48, 90, 62, 150, 200]
})
orders

,Month,Category,Region,Revenue,Units
0,Jan,Electronics,US,5000,50
1,Jan,Clothing,EU,1200,120
2,Feb,Electronics,US,4800,48
3,Feb,Clothing,US,900,90
4,Mar,Electronics,EU,6200,62
5,Mar,Clothing,EU,1500,150
6,Mar,Food,US,800,200


In [10]:
# Monthly revenue by Category
monthly = pd.pivot_table(orders,
    values='Revenue',
    index='Month',
    columns='Category',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)
monthly

Category,Clothing,Electronics,Food,Total
Month,,,,
Feb,900,4800,0,5700
Jan,1200,5000,0,6200
Mar,1500,6200,800,8500
Total,3600,16000,800,20400


In [11]:
# Multi-value pivot: revenue AND units simultaneously
multi = pd.pivot_table(orders,
    values=['Revenue', 'Units'],
    index='Category',
    columns='Region',
    aggfunc='sum',
    fill_value=0
)
multi

Revenue       Units     
Region           EU    US    EU   US
Category                            
Clothing       2700   900   270   90
Electronics    6200  9800    62   98
Food              0   800     0  200

In [12]:
# Dict aggfunc — different aggregations per column
summary = pd.pivot_table(orders,
    values={'Revenue': 'sum', 'Units': 'mean'},
    index='Category',
    columns='Region',
    fill_value=0
)
summary

Revenue          Units       
Region           EU      US     EU     US
Category                                 
Clothing     1350.0   900.0  135.0   90.0
Electronics  6200.0  4900.0   62.0   49.0
Food            0.0   800.0    0.0  200.0

## `pivot_table` vs `groupby`

|Aspect|	pivot_table|	groupby|
|--|--|--|
|Output shape|	Wide (cross-tabular)|	Long (stacked)|
|Axes control|	Row + Column headers|	Single grouping axis|
|Best for|	Comparing across two dimensions	|Aggregating along one dimension|
|Handles missing|	fill_value / NaN|	No auto-fill|

### Common Gotchas

`aggfunc='count'` counts non-null entries, not total rows. Use 'size' for total row count including nulls.




In [13]:
# count vs size
pd.pivot_table(product_region_sales_df, values='Sales', index='Region', aggfunc='count')   # non-null only


,Sales
Region,
East,1
North,2
South,2


In [18]:
pd.pivot_table(product_region_sales_df, values='Sales', index='Region', aggfunc=np.size)    # all rows

,Sales
Region,
East,1
North,2
South,2


## `pd.melt()`

(2026-07-20T13:23:36+07:00 - AI Generated with Claude Sonnet 4.6 (Thinking))

`pd.melt()` is the inverse of pivot — it transforms a DataFrame from wide format to long format by "unpivoting" columns into rows.

### Syntax

```python
pd.melt(frame, id_vars=None, value_vars=None, var_name='variable', value_name='value', col_level=None)
```

|Parameter|	Description|
|--|--|
|frame|	The DataFrame to melt|
|id_vars|	Columns to keep as identifiers (not melted)|
|value_vars|	Columns to melt (defaults to all non-id columns)|
|var_name|	Name for the new column holding old column names|
|value_name|	Name for the new column holding the values|

### Basic example

In [20]:
school_subjects_df = pd.DataFrame({
    'name': ['Alice', 'Bob'],
    'math': [90, 85],
    'science': [88, 92]
})
school_subjects_df

,name,math,science
0,Alice,90,88
1,Bob,85,92


In [21]:
pd.melt(school_subjects_df, id_vars='name', value_vars=['math', 'science'], var_name='subject', value_name='score')

,name,subject,score
0,Alice,math,90
1,Bob,math,85
2,Alice,science,88
3,Bob,science,92
